# LifeLink Phase 5 — deep-model training (Colab)

Trains the existing **ANN**, **LSTM** and **Autoencoder** pipelines, evaluates them, verifies that the `.keras` artifacts load and run inference, and packages them with SHA-256 checksums.

**Honesty rules**
- All datasets are deterministic **synthetic** data generated by the existing LifeLink scripts. Metrics are not hospital performance.
- The notebook reports the actual device (GPU or CPU). It never claims GPU use when TensorFlow sees none.
- It does not create placeholder artifacts. If training fails, the verifier exits non-zero.
- No API keys, passwords, tokens or real donor/hospital data belong in this notebook.

Expected runtime: a few minutes on CPU for these small synthetic datasets. A GPU is optional.

## 1. Provide the project

Upload a source archive that contains `ml_service/` (the Phase 5 release archive is fine). Alternatively, mount Drive and set `PROJECT_DIR` to an extracted copy. Do not upload a `.env` file or credentials.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/lifelink')
if not PROJECT_DIR.exists():
    from google.colab import files
    uploaded = files.upload()
    archive = next(iter(uploaded))
    import zipfile
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(PROJECT_DIR)

candidates = [PROJECT_DIR, *PROJECT_DIR.iterdir()]
ml_root = next((p / 'ml_service' for p in candidates if (p / 'ml_service' / 'scripts' / 'train_all_deep.py').exists()), None)
assert ml_root, 'ml_service/scripts/train_all_deep.py was not found in the uploaded archive.'
os.chdir(ml_root.parent)
print('Project root:', Path.cwd())
print('ML service:', ml_root)
print('Existing classical artifacts preserved:', sorted(p.name for p in (ml_root / 'artifacts').glob('*') if p.is_file())[:8])

## 2. Install compatible deep-learning dependencies

`requirements-deep.txt` pins TensorFlow CPU 2.17.1 and Keras 3.5.0, compatible with Python 3.10–3.12. TensorFlow CPU is used even on a GPU runtime in this pinned setup; the runtime check below reports the truth. If you intentionally want GPU wheels, change the pin deliberately and record the version in the metadata.

In [ ]:
%pip install -q -r ml_service/requirements-deep.txt
import tensorflow as tf
import keras
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpus if gpus else 'none — training will run on CPU')
assert tf.__version__.startswith('2.'), tf.__version__

## 3. Train all three deep models

This calls the existing `train_all_deep()` implementation. It preserves the three classical artifacts and writes new versioned files under `ml_service/artifacts/deep/`: `.keras`, `.scaler.joblib`, `.history.json`, `.metadata.json`.

In [ ]:
import time, subprocess, sys
started = time.time()
result = subprocess.run([sys.executable, 'scripts/train_all_deep.py'], cwd='ml_service', text=True, capture_output=True)
print(result.stdout[-12000:])
print(result.stderr[-4000:])
print(f'Training command exited {result.returncode} in {time.time() - started:.1f}s')
assert result.returncode == 0, 'Training failed; no artifacts should be integrated.'

## 4. Report stored metrics

`evaluate_all.py` only reads metadata. It does not retrain or impute missing numbers.

In [ ]:
result = subprocess.run([sys.executable, 'scripts/evaluate_all.py'], cwd='ml_service', text=True, capture_output=True)
print(result.stdout)
print(result.stderr)
print('Evaluation command exit:', result.returncode)
assert result.returncode == 0, 'One or more deep artifacts are missing.'

## 5. Load, infer, checksum and package

The verifier loads each `.keras` model and its scaler, runs a deterministic one-row inference, validates the metadata, records SHA-256 checksums and creates `lifelink-deep-artifacts.zip`. A missing or unloadable model exits non-zero.

In [ ]:
package = Path('/content/lifelink-deep-artifacts.zip')
result = subprocess.run(
    [sys.executable, 'scripts/verify_deep_artifacts.py', '--package', str(package)],
    cwd='ml_service', text=True, capture_output=True,
)
print(result.stdout[-16000:])
print(result.stderr[-4000:])
assert result.returncode == 0, 'Artifact verification failed. Do not integrate this package.'
print('Verified package:', package, package.stat().st_size, 'bytes')

In [ ]:
import hashlib, json
print('SHA-256:', hashlib.sha256(package.read_bytes()).hexdigest())
manifest = json.loads((ml_root / 'artifacts' / 'deep' / 'deep-artifacts-manifest.json').read_text())
print(json.dumps({k: v.get('inference') for k, v in manifest['models'].items()}, indent=2))
from google.colab import files
files.download(str(package))

## 6. Integrate the verified package locally

1. Download `lifelink-deep-artifacts.zip` and its printed SHA-256.
2. Extract it into `ml_service/artifacts/` so the files land in `ml_service/artifacts/deep/`.
3. Run `python scripts/verify_deep_artifacts.py` again in an environment with the same TensorFlow version.
4. Run the FastAPI tests and `GET /ai/models`; ANN, LSTM and Autoencoder must report `ready: true` with the same versions.
5. Do not integrate if the checksum, load test or metadata validation fails.

The three classical models (`demand_model.json`, `donor_response.joblib`, `inventory_anomaly.joblib`) are never rewritten by this notebook.